In [1]:
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")

from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n_M import BeliefMDP_n_M_Mapping
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time
import warnings

warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Generate p_n^{(M)} transition probability matrix for BeliefMDP_n_M_Mapping.

This notebook:
1. Creates a BeliefMDP_n_M_Mapping instance with coarse quantization (n=2)
2. Automatically computes and caches p_n^{(M)} for all belief-action pairs
3. Verifies the computed matrix properties

For Mapping:
- Belief space: π(m) - shape (len_M,)
- Quantized belief space: Π_n^(M) with N_n = len_M
- Pose is known (not part of belief state)
- η_n signature: η_n(π_new, π, x_current, u, x_next, ...)
"""


CWD: /global/home/hpc5656/SLAM
CUDA_PATH not set, attempting to load modules...
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH to include: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)
✓ ctypes.util.find_library('nvrtc') found: libnvrtc.so.12
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)


'\nGenerate p_n^{(M)} transition probability matrix for BeliefMDP_n_M_Mapping.\n\nThis notebook:\n1. Creates a BeliefMDP_n_M_Mapping instance with coarse quantization (n=2)\n2. Automatically computes and caches p_n^{(M)} for all belief-action pairs\n3. Verifies the computed matrix properties\n\nFor Mapping:\n- Belief space: π(m) - shape (len_M,)\n- Quantized belief space: Π_n^(M) with N_n = len_M\n- Pose is known (not part of belief state)\n- η_n signature: η_n(π_new, π, x_current, u, x_next, ...)\n'

In [2]:
# Configuration
n = 2  # Coarse quantization for initial testing
M = 3  # Belief space quantization parameter
beta = 0.95  # Discount factor

print(f"Configuration:")
print(f"  n (state quantization): {n}")
print(f"  M (belief quantization): {M}")
print(f"  β (discount factor): {beta}")
print(f"\nThis will create a BeliefMDP_n_M_Mapping instance which will:")
print(f"  1. Load or generate belief codebook (Π_n^M) with N_n = len_M")
print(f"  2. Load cached T_mat and p_n^{(M)}")
print(f"  3. Compute c_n^{(M)} for all belief-state-action combinations")
print(f"  4. Cache the result for future use")

# Load environment configuration
obstacles, area = load_obstacles_config(environment='toy2')

# Create models
motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
sensor = LIDAR(fov=360, r_max=10.0, B=8)
grid_map = LidarGridMapVec(
    x_min=area[0], x_max=area[1],
    y_min=area[2], y_max=area[3],
    quantization_level=n,
)

print(f"Environment setup:")
print(f"  Map bounds: x=[{area[0]}, {area[1]}], y=[{area[2]}, {area[3]}]")
print(f"  Map quantization: {n}x{n} = {n**2} cells")
print(f"  Total maps: 2^{n**2} = {2**(n**2)}")
print(f"  Motion model: DoubleIntegratorModel (dt={motion_model.dt}, max_a={motion_model.max_a})")
print(f"  Sensor: LIDAR (fov={sensor.fov}°, r_max={sensor.r_max}, B={sensor.B})")


Configuration:
  n (state quantization): 2
  M (belief quantization): 3
  β (discount factor): 0.95

This will create a BeliefMDP_n_M_Mapping instance which will:
  1. Load or generate belief codebook (Π_n^M) with N_n = len_M
  2. Load cached T_mat and p_n^3
  3. Compute c_n^3 for all belief-state-action combinations
  4. Cache the result for future use
Environment setup:
  Map bounds: x=[0, 10], y=[0, 10]
  Map quantization: 2x2 = 4 cells
  Total maps: 2^4 = 16
  Motion model: DoubleIntegratorModel (dt=1.0, max_a=2.0)
  Sensor: LIDAR (fov=6.283185307179586°, r_max=10.0, B=8)


In [3]:
# Create BeliefMDP_n_M_Mapping instance
print("="*70)
print("Creating BeliefMDP_n_M_Mapping instance...")
print("="*70)
print()

start_time = time.time()

bmdp_M = BeliefMDP_n_M_Mapping(
    M=M,
    β=beta,
    n=n,
    motion_model=motion_model,
    measurement_model=sensor,
    obstacles=obstacles,
    _map=grid_map,
    sigma_v=1.0,
    j_batch_size=816,
    i_batch_size=1
)

# Set known pose and seed map
known_pose = np.array([5.0, 5.0, 0.0, 0.0])  # [px, py, vx, vy]
bmdp_M.known_pose = known_pose
bmdp_M.map.seed_from_obstacles(obstacles)

elapsed_time = time.time() - start_time

print()
print("="*70)
print("Initialization complete!")
print("="*70)
print(f"Total time: {elapsed_time:.2f}s")
print()
print(f"Belief space statistics:")
print(f"  Cardinality |Π_n^M|: {bmdp_M.BQ.cardinality:,}")
print(f"  Belief dimension N_n: {bmdp_M.len_M} (for mapping, N_n = len_M)")
print(f"  State space size m_n: {bmdp_M.SQ.m_n}")
print(f"  Map space size len_M: {bmdp_M.len_M}")
print(f"  Action space size n_u: {bmdp_M.AQ.n_u}")
print(f"  c_n^{(M)} shape: {bmdp_M.c_n_M.shape}")
print(f"  c_n^{(M)} dtype: {bmdp_M.c_n_M.dtype}")


Creating BeliefMDP_n_M_Mapping instance...

Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_ea380e8a.npz
  ✓ Cache validation passed
  ✓ Loaded codebook from cache: cache/belief_quantizer/belief_quantizer_M3_N16.npz
Looking for p_n_M cache at: /global/home/hpc5656/SLAM/cache/MAP/p_n_M_mapping/p_n_M_mapping_M3_n2_map2x2_max2.0_e5f2a9fd.npz
Cache file exists: False
Cache file does not exist: /global/home/hpc5656/SLAM/cache/MAP/p_n_M_mapping/p_n_M_mapping_M3_n2_map2x2_max2.0_e5f2a9fd.npz
Found 1 potential cache files, trying to load...
  Trying: p_n_M_mapping_M3_n2_map2x2_max2.0_2e7b2bcb.npz
  ✓ Successfully loaded: p_n_M_mapping_M3_n2_map2x2_max2.0_2e7b2bcb.npz
Loaded cached p_n_M from /global/home/hpc5656/SLAM/cache/MAP/p_n_M_mapping/p_n_M_mapping_M3_n2_map2x2_max2.0_e5f2a9fd.npz
Computing c_n_M for the first time...
This will compute costs for 816 beliefs × 16 states × 4 actions
  Computing costs for 816 beliefs × 16 states × 4 actions...
Saved c_n

In [4]:
# Verify c_n^{(M)} properties
print("="*70)
print("Verifying c_n^{(M)} properties...")
print("="*70)
print()

c_n_M = bmdp_M.c_n_M
cardinality, m_n, n_u = c_n_M.shape

print(f"Shape: ({cardinality}, {m_n}, {n_u})")
print(f"  - {cardinality:,} beliefs")
print(f"  - {m_n} states")
print(f"  - {n_u} actions")
print(f"  - Total entries: {c_n_M.size:,}")
print()

# Value statistics
min_val = float(np.min(c_n_M))
max_val = float(np.max(c_n_M))
mean_val = float(np.mean(c_n_M))
std_val = float(np.std(c_n_M))

print(f"Value statistics:")
print(f"  Min: {min_val:.6f}")
print(f"  Max: {max_val:.6f}")
print(f"  Mean: {mean_val:.6f}")
print(f"  Std: {std_val:.6f}")
print()

if min_val < 0:
    print(f"  ⚠ WARNING: Found negative values!")
else:
    print(f"  ✓ All values are non-negative")
print()

# Check for NaN or Inf
has_nan = bool(np.any(np.isnan(c_n_M)))
has_inf = bool(np.any(np.isinf(c_n_M)))

if has_nan:
    print(f"  ⚠ WARNING: Found NaN values!")
else:
    print(f"  ✓ No NaN values")

if has_inf:
    print(f"  ⚠ WARNING: Found Inf values!")
else:
    print(f"  ✓ No Inf values")
print()

print("="*70)
print("Verification complete!")
print("="*70)


Verifying c_n^{(M)} properties...

Shape: (816, 16, 4)
  - 816 beliefs
  - 16 states
  - 4 actions
  - Total entries: 52,224

Value statistics:
  Min: 1.322876
  Max: 2.322876
  Mean: 1.590533
  Std: 0.330848

  ✓ All values are non-negative

  ✓ No NaN values
  ✓ No Inf values

Verification complete!
